<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-classical-and-quantum-physics/blob/main/notebooks/lecture-08/exercise-01-rossler-oscillators.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

<hr/>
# Deep Learning in Classical and Quantum Physics
2026/2027 <br>
**Author:** Marcin Płodzień <br>
**Exercise:** 01 <br>
**Related labs:** 033, 034, 035, 036
<hr/>

# Exercise 01 — Rössler system and superimposed oscillators

In Lectures 7 and 8 we characterised three signals and forecast them with linear models, NVAR (nonlinear vector autoregression) and recurrent neural networks. In this exercise we apply the same toolbox to two new signals taken from `reservoirpy.datasets`, and we analyse each of them before we forecast it, as in the labs.

The first signal is a sum of eight sine waves with incommensurate frequencies, the multiple superimposed oscillator (MSO) benchmark of reservoir computing. Its motion is quasi-periodic, with a finite number of oscillation modes. We locate its spectral lines, show that an autoregressive (AR) model of order $2K$ for $K$ sinusoids forecasts it almost perfectly, and explain this with the Koopman picture of Lab 035. The second signal is the $x$ coordinate of the Rössler system, a three-dimensional chaotic flow with a single nonlinear term. We study its physics, spectrum, autocorrelation function (ACF), delay embedding, Lyapunov spectrum and Kaplan–Yorke dimension $D_{\rm KY}$. In Section 3 we forecast both signals with persistence, AR, the moving-average (MA) model, NVAR, a recurrent neural network (RNN) and a long short-term memory (LSTM) network, and read the results against the leading Lyapunov exponent $\lambda_1$ and $D_{\rm KY}$.

Each exercise states what to compute and what to report, and names the lab whose functions and conventions it reuses. The protocol and the scores are those of Lab 036 and are defined in the first code cell. Two exercises are marked as optional. The surrogate test of both signals closes the analysis part, and an echo state network (ESN) closes the forecasting part. No other exercise uses their results.

> **How to run this notebook in Google Colab**
> 1. Go to <https://colab.research.google.com> and upload the notebook with *File → Upload notebook*. The first code cell sets `JAX_PLATFORMS=cpu`.
> 2. `jax`, `flax`, `optax`, `scipy`, `statsmodels`, `pandas` and `matplotlib` are pre-installed in Colab. Install ReservoirPy with `pip install reservoirpy`.
> 3. Run the cells in order. Later cells use variables defined earlier.

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"     # run JAX on the CPU (set before importing jax)

import json, time
from functools import partial
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import scipy.signal
from statsmodels.tsa.stattools import acf as sm_acf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
import warnings

import jax
jax.config.update("jax_enable_x64", True)   # signals, spectra, exponents and ridge regressions in float64
import jax.numpy as jnp
import optax
import flax
from flax import nnx
import reservoirpy
from reservoirpy.datasets import mso, mso2, mso8, rossler
from reservoirpy.nodes import NVAR, Reservoir, Ridge as RpyRidge

# common style of the notebook figures
mpl.rcParams.update({"font.size": 14, "axes.labelsize": 15, "axes.titlesize": 15,
                     "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 12,
                     "figure.titlesize": 16, "lines.linewidth": 2.0,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
SLIDE_RC = {"font.size": 11, "axes.labelsize": 11, "axes.titlesize": 11, "xtick.labelsize": 11,
            "ytick.labelsize": 11, "legend.fontsize": 11, "lines.linewidth": 1.4, "figure.dpi": 150}
UJBlue, UJNavy, DSOrange, UJGray = "#005CA7", "#002B5C", "#D35A05", "#5A6270"
STYLE = {"persistence": dict(color="0.55", ls=":"), "AR": dict(color="0.35", ls="--"),
         "AR recursive": dict(color=UJBlue, ls="-."), "MA": dict(color="#8172B2", ls="-."),
         "NVAR": dict(color=DSOrange, ls="--"), "RNN": dict(color=UJBlue, ls="-"),
         "LSTM": dict(color="#C44E52", ls="-"), "ESN": dict(color="#8C564B", ls="-.")}   # as in Lab 036

OUT = "figures"
os.makedirs(OUT, exist_ok=True)
def save(fig, name):
    # figures are drawn at their printed size and saved as PDF and PNG
    for ext in ("pdf", "png"):
        fig.savefig(f"{OUT}/{name}.{ext}", dpi=300, bbox_inches="tight", pad_inches=0.02)

# Common protocol of Lectures 7 and 8 (as in Lab 036): N = 4000 samples, training block = first 3000
# (its last 500 = validation), test block = last 1000, rescaling to [0, 1] with the training min/max,
# leads tau = 1..50, input windows of at most L = 64 past values.
N_SAMPLES, N_TRAIN, N_VAL, TAU_MAX, L_WIN = 4000, 3000, 500, 50, 64
TAUS = np.arange(1, TAU_MAX + 1)
K_FIT = np.arange(L_WIN - 1, N_TRAIN - N_VAL - TAU_MAX)      # fitting origins 63..2449
K_VAL = np.arange(N_TRAIN - N_VAL - 1, N_TRAIN - TAU_MAX)    # validation origins 2499..2949
K_TEST = np.arange(N_TRAIN - 1, N_SAMPLES - TAU_MAX)         # test origins 2999..3949

def rescale01(s, n_train=N_TRAIN):
    lo, hi = s[:n_train].min(), s[:n_train].max()
    return (s - lo) / (hi - lo)

def windows(s, ks, L=L_WIN):
    # inputs U[i] = (s_{k-L+1}, ..., s_k) and targets Y[i] = (s_{k+1}, ..., s_{k+50}) for k = ks[i]
    U = s[ks[:, None] + np.arange(-L + 1, 1)[None, :]]
    Y = s[ks[:, None] + TAUS[None, :]]
    return U, Y

@jax.jit
def nmse_and_r(pred, true):
    # pred, true: (n, TAU_MAX); averages over the n origins, one value per lead
    err = jnp.mean((pred - true) ** 2, axis=0) / jnp.var(true, axis=0)
    pc, tc = pred - pred.mean(0), true - true.mean(0)
    r = jnp.sum(pc * tc, 0) / jnp.sqrt(jnp.sum(pc ** 2, 0) * jnp.sum(tc ** 2, 0))
    return err, r

def score(pred, true):
    # NMSE(tau), r(tau), C(tau) = r^2, C_tot = sum_tau C(tau) and the skill horizon tau_1/2 (first tau with C < 0.5)
    e, r = (np.asarray(a) for a in nmse_and_r(jnp.asarray(pred, jnp.float64), jnp.asarray(true, jnp.float64)))
    C = r ** 2
    below = np.nonzero(C < 0.5)[0]
    return dict(nmse=e, r=r, C=C, Ctot=float(C.sum()), tau_half=int(below[0] + 1) if len(below) else None)

nums = {}
t_start = time.time()
print("jax", jax.__version__, "| flax", flax.__version__, "| reservoirpy", reservoirpy.__version__,
      "| devices", jax.devices())

<hr/>
## 1. Multiple superimposed oscillators

The MSO signal is a sum of $K$ sine waves sampled at integer times $n=0,1,\dots,N-1$,
$$
s_n=\sum_{i=1}^{K}\sin(\omega_i n),
$$
where $\omega_i$ are angular frequencies in radians per sample. The cyclic frequency of the $i$-th wave is $f_i=\omega_i/2\pi$ in cycles per sample. Jaeger introduced the two-frequency version as a benchmark that a standard echo state network fails to continue in free run [H. Jaeger, M. Lukoševičius, D. Popovici and U. Siewert, Neural Netw. **20**, 335 (2007), [doi:10.1016/j.neunet.2007.04.016](https://doi.org/10.1016/j.neunet.2007.04.016)], and Roeschies and Igel extended it to eight frequencies [B. Roeschies and C. Igel, Logic J. IGPL **18**, 635 (2010), [doi:10.1093/jigpal/jzp043](https://doi.org/10.1093/jigpal/jzp043)]. ReservoirPy provides the generator `mso(n_timesteps, freqs, normalize=True)` and the two presets `mso2` with $\omega=(0.2,0.311)$ and `mso8` with
$$
\boldsymbol\omega=(0.2,\ 0.311,\ 0.42,\ 0.51,\ 0.63,\ 0.74,\ 0.85,\ 0.97).
$$
With `normalize=True` the library rescales the whole record to $[-1,1]$ with its own minimum and maximum, which uses the test block. We therefore generate the signal with `normalize=False` and apply the protocol rescaling to $[0,1]$ fitted on the training block.

### Exercise 1.1. Signal and sampling

Generate $N=4000$ samples of `mso8` with `normalize=False`. Check them against the formula above, apply the protocol rescaling, and plot the whole record and a zoom on the first 300 samples. Make a table of $\omega_i$, $f_i$ and the period $1/f_i$ in samples. Compute the Nyquist frequency $f_{\rm Ny}=1/(2\Delta t)$ with $\Delta t=1$ sample, the number of samples per period of the fastest wave and the frequency resolution $1/(N\Delta t)$ of the record, and state whether the sampling resolves all eight waves. The frequency ratios are irrational, so the signal never repeats. Check this by computing the mean mismatch $\langle|s_{n+T}-s_n|\rangle$ for all shifts $T=20,\dots,2000$ and comparing its minimum with the standard deviation of the signal.

In [ ]:
# your code: generate mso8 without normalisation, check it against the formula, rescale, plot,
# tabulate omega_i, f_i and periods, sampling numbers, and the shift mismatch

### Exercise 1.2. Power spectrum and spectral lines

Compute the one-sided periodogram of the record without a window and with the Hann window, and the Welch estimate with segments of $L=1024$ samples (functions `periodogram_jax`, `windowed_periodogram` and `welch_jax` of Lab 033, checked against `scipy.signal`). Plot the three on a logarithmic scale with the true frequencies $f_i$ marked. Find the eight spectral peaks of the Hann periodogram with `spectral_peaks` of Lab 033 and compare their frequencies with $f_i$ in units of the resolution $\Delta f=1/N$. A peak located on the grid $f_q=q/N$ is uncertain by $\Delta f/2$. Refine each peak by fitting a parabola through $\ln\hat S$ at the three bins $q-1$, $q$, $q+1$, whose vertex lies at $q+\delta$ with
$$
\delta=\frac{1}{2}\,\frac{\ln\hat S_{q-1}-\ln\hat S_{q+1}}{\ln\hat S_{q-1}-2\ln\hat S_{q}+\ln\hat S_{q+1}},
$$
and report how much the refinement reduces the error. Explain the role of the window for this signal.

In [ ]:
# your code: periodogram (rectangular and Hann), Welch with L = 1024, eight peaks of the Hann periodogram,
# parabolic refinement, table of errors in units of 1/N, figure

### Exercise 1.3. Autocorrelation function

Compute the ACF $\hat\rho(k)$ for lags $k=0,\dots,400$ with `acf_jax` of Lab 033 and check it against `statsmodels`. For equal amplitudes and a long record the time average of $\sin(\omega_in)\sin(\omega_jn+\omega_jk)$ vanishes for $i\neq j$ and equals $\tfrac12\cos(\omega_ik)$ for $i=j$, which gives
$$
\rho(k)=\frac{1}{K}\sum_{i=1}^{K}\cos(\omega_ik).
$$
The estimator $\hat\rho$ sums $N-k$ products and divides by $N$, so compare it with $(1-k/N)\,\rho(k)$. Plot both, report the largest difference, and compute the correlation times of Lab 033 (first zero, first drop below $1/e$, envelope below $1/e$). Explain why the envelope of a quasi-periodic ACF does not decay to zero.

In [ ]:
# your code: ACF in JAX vs statsmodels, comparison with (1 - k/N) rho(k), correlation times, figure

### Exercise 1.4. Autoregressive model and Koopman eigenvalues

A single sine wave obeys an exact linear recursion. From $\sin(\omega(n+1))+\sin(\omega(n-1))=2\cos\omega\,\sin(\omega n)$,
$$
s_{n+1}=2\cos\omega\;s_n-s_{n-1},
$$
an AR(2) model without noise whose characteristic roots are $e^{\pm i\omega}$. A sum of $K$ sine waves with distinct frequencies obeys the AR($2K$) recursion whose characteristic polynomial is the product of the $K$ factors $1-2\cos\omega_i\,z+z^2$. In the language of Lab 035, each wave is spanned by a pair of Koopman eigenfunctions with eigenvalues $e^{\pm i\omega_i}$, the delay vector of length $2K$ is a dictionary that is invariant under the dynamics, and the AR coefficients form the last row of its companion matrix.

Verify this numerically. Fit AR($p$) models with intercept by least squares (Lab 034) on the samples before the validation block, for $p=1,\dots,24$, and compute the one-step NMSE (normalised mean-square error) on the validation block, for `mso2` ($K=2$) and `mso8` ($K=8$), both rescaled with the protocol. Plot NMSE against $p$ and locate the drop. Then fit AR(16) to `mso8` on the whole training block, check the coefficients against `statsmodels` `AutoReg`, expand the product polynomial above with `numpy.polynomial` and compare its coefficients with the fitted ones. Compute the eigenvalues of the companion matrix (Lab 035), plot them in the complex plane with the unit circle, and compare their moduli and angles with $1$ and $\omega_i$.

In [ ]:
# your code: validation NMSE of AR(p), p = 1..24, for mso2 and mso8; AR(16) coefficients (JAX and statsmodels)
# against the product polynomial; companion-matrix eigenvalues against exp(+-i omega_i); figure

### Exercise 1.5. Forecasts of the MSO signal

Forecast the rescaled `mso8` signal with the protocol of Lab 036 at the test origins $k=2999,\dots,3949$ and leads $\tau=1,\dots,50$. Use persistence, $\hat s_{k+\tau}=s_k$, the direct AR model of Lab 036 (one ridge readout per lead, order $p$ and ridge strength chosen by the validation $C_{\rm tot}$), and the recursive AR(16) model of Exercise 1.4 iterated from every test origin (Lab 034). Report $C_{\rm tot}=\sum_\tau C(\tau)$ with $C(\tau)=r(\tau)^2$, the skill horizon $\tau_{1/2}$ and the NMSE at $\tau=1$ and $\tau=50$ for each model, and plot $C(\tau)$. Finally run AR(16) freely from the first test origin over the whole test block and plot it on top of the data.

In [ ]:
# your code: persistence, direct AR (ridge, selection on the validation block), recursive AR(16) from every test origin,
# scores, C(tau) figure, free run over the test block

<hr/>
## 2. Rössler system

Rössler looked for the simplest autonomous flow with chaotic motion and found a system with three variables and a single nonlinear term [O. E. Rössler, Phys. Lett. A **57**, 397 (1976), [doi:10.1016/0375-9601(76)90101-8](https://doi.org/10.1016/0375-9601(76)90101-8)],
$$
\dot x=-y-z,\qquad \dot y=x+a\,y,\qquad \dot z=b+z\,(x-c).
$$
The linear part in the $(x,y)$ plane is an oscillator with negative damping for $a>0$, so trajectories near the plane spiral outwards. The variable $z$ stays small until $x$ exceeds $c$. Then the product $zx$ makes $z$ grow quickly, and the term $-z$ in $\dot x$ throws the trajectory back towards the centre of the spiral, where the outward motion starts again. The stretching happens on the spiral and the folding in the excursion in $z$, the same two ingredients as in the logistic map of Lab 033. We use the standard parameters $a=0.2$, $b=0.2$, $c=5.7$, which are the defaults of ReservoirPy, and we observe only $x(t)$.

### Exercise 2.1. Equations, fixed points and the generated signal

Read the source of `reservoirpy.datasets.rossler` (for example with `inspect.getsource`). State its default parameters, initial state, sampling step `h` and the integrator it calls. Check how the output times are constructed from `n_timesteps` and `h`, and find the value of `h` that gives a sampling interval of exactly $\Delta t=0.1$. Generate the trajectory with this sampling interval and tolerances `rtol = atol = 1e-10` passed to the integrator, discard the first 500 time units as a transient, and keep $N=4000$ samples. Plot the attractor in $(x,y,z)$ and $x(t)$.

Find the fixed points of the flow analytically, check $\mathbf F(\mathbf u^*)=0$ numerically, and compute the eigenvalues of the Jacobian $J=\partial\mathbf F/\partial\mathbf u$ with `jax.jacfwd` (as for Lorenz-63 in Lab 033). From the complex pair at the fixed point near the origin derive the rotation frequency of the spiral in cycles per time unit. Finally write the right-hand side in JAX, integrate it with RK4 at step $h=0.01$ from the first stored state, and compare with the ReservoirPy trajectory over the first 20 time units.

In [ ]:
# your code: defaults and time grid of reservoirpy.datasets.rossler, generation with dt = 0.1 and transient removed,
# fixed points and Jacobian eigenvalues with jax.jacfwd, JAX RK4 check, attractor and x(t) figure

### Exercise 2.2. Sampling and power spectrum

Compute the Nyquist frequency and the frequency resolution of the record in cycles per time unit, and the number of samples per rotation. Estimate the PSD (power spectral density) with the Hann-windowed periodogram of the whole record and with Welch segments of $L=1024$ samples (Lab 033), and plot both. Locate the dominant peak $f_0$ and the peaks near its harmonics $2f_0$ and $3f_0$. Compare $f_0$ with the rotation frequency at the inner fixed point of Exercise 2.1 and with the mean rotation frequency $\langle\dot\theta\rangle/2\pi$ of the phase $\theta=\arctan(y/x)$ (use `np.unwrap` of `np.arctan2(y, x)` on the full state). Measure the full width at half maximum of the dominant peak in the Hann periodogram and compare it with the width of the Hann peak of a pure sine wave of the same frequency and length. Describe the spectrum in the terms of Lab 033 (lines, broad peaks, broadband).

In [ ]:
# your code: Nyquist and resolution, Hann periodogram and Welch, dominant peak and harmonics, comparison with the
# rotation frequencies, peak width against a pure sine wave, figure

### Exercise 2.3. Autocorrelation function

Compute $\hat\rho(k)$ of Rössler $x$ for lags up to 1000 samples (100 time units) with `acf_jax` and the three correlation times of Lab 033, in samples and in time units. Find the lag of the first local maximum of $\hat\rho$ and compare it with $1/f_0$. Plot $\hat\rho$ with its envelope and explain the slow decay of the envelope with the result of Exercise 2.2.

In [ ]:
# your code: ACF up to lag 1000, correlation times, first maximum against 1/f0, figure

### Exercise 2.4. Delay embedding

Choose the delay $\ell$ as the first minimum of the mutual information $I(\ell)$ with 16 bins and the embedding dimension $m$ as the smallest dimension at which the ratio test of false nearest neighbours (FNN) falls below 5%, with $R_{\rm tol}=15$ and a Theiler window of 50 samples (functions `mutual_information`, `delay_embed`, `nearest_neighbour` and `fnn_fractions` of Lab 033). Compare $\ell$ with the first zero of the ACF and with a quarter of the rotation period. Plot $I(\ell)$, the FNN fractions and a three-dimensional delay reconstruction next to the true attractor.

In [ ]:
# your code: mutual information and its first minimum, FNN fractions and m, delay reconstruction figure

### Exercise 2.5. Lyapunov spectrum from the equations

Compute the three Lyapunov exponents of the flow with the Benettin QR algorithm of Lab 033. Propagate three orthonormal tangent vectors with the Jacobian $D\boldsymbol\Phi_h$ of one RK4 step ($h=0.01$) obtained with `jax.jacfwd`, re-orthonormalise every $T_r=0.1$ time units, and average $\ln|R_{ii}|$ over 20000 time units, starting from the last stored state. Estimate the standard errors from ten segments of 2000 time units. The sum of the exponents equals the mean phase-space contraction rate. For the Rössler flow $\nabla\cdot\mathbf F=a+x-c$ depends on the position, so compare $\sum_i\lambda_i$ with the time average $a-c+\langle x\rangle$ along the same trajectory. Plot the running averages of the exponents against the averaging time.

In [ ]:
# your code: Benettin QR with jax.jacfwd of the RK4 step, exponents and standard errors, sum rule, convergence figure

### Exercise 2.6. Exponent from data, Kaplan–Yorke dimension and prediction horizon

Estimate $\lambda_1$ from the measured $x$ alone with the method of Rosenstein *et al.* (function `rosenstein_curve` of Lab 033) with the embedding of Exercise 2.4, the Theiler window of 50 samples and $k$ up to 400 samples. Plot $\langle\ln d(k)\rangle$, choose a fit range in the linear part of the curve, and report the slope per sample and per time unit for a few fit ranges next to the Benettin value.

Compute the Kaplan–Yorke dimension $D_{\rm KY}=j+\Sigma_j/|\lambda_{j+1}|$ (function `kaplan_yorke` of Lab 033, with exponents below three standard errors set to zero). Compute the Lyapunov time $T_\lambda=1/\lambda_1$ in time units and in samples, the mean error growth factor $e^{\lambda_1\cdot50\Delta t}$ over the longest forecast lead, and the prediction horizon $T_{\rm pred}=\lambda_1^{-1}\ln(\Delta/\delta_0)$ for $\delta_0=10^{-6}$ and $\Delta=1$. Check $T_{\rm pred}$ with 100 pairs of trajectories that start $10^{-6}$ apart, as in Lab 033.

In [ ]:
# your code: Rosenstein curve and slopes, D_KY, Lyapunov time, error growth over 50 samples, T_pred and the twin test

### Optional exercise 2.7. Surrogate tests

Apply the surrogate test of Lab 033 (Appendix A) to both signals. The null hypothesis is a white Gaussian noise passed through a linear filter and observed through a static monotone function. Generate 19 IAAFT (iterative amplitude-adjusted Fourier transform) surrogates of each signal, use the normalised error $E$ of the nearest-neighbour forecast one step ahead as the statistic with a Theiler window of 50 samples, and reject the null hypothesis when the data error is the smallest of the 20 values. Use delay vectors of $m=16$ entries and delay $\ell=1$ for `mso8`, the window of the exact AR(16) model, and the embedding $(m,\ell)$ of Exercise 2.4 for Rössler $x$. Report the data error, the range of the surrogate errors, $z=(E_{\rm data}-\bar E_{\rm surr})/\sigma_{\rm surr}$ and the rank, and the fraction of the total periodogram power in the eight largest bins.

Repeat the test for a control signal, the sum of eight sine waves whose frequencies are moved to the nearest Fourier frequencies $q/N$ of the record. Interpret the three outcomes. Keep in mind what a phase-randomised surrogate does to a spectral line that falls between two Fourier frequencies, and that a deterministic signal obeying an exact linear recursion lies outside the null hypothesis.

In [ ]:
# your code: IAAFT surrogates, nearest-neighbour prediction error, rank test for mso8 (m = 16, l = 1),
# for the control signal with frequencies on the Fourier grid, and for Rossler x with the embedding of Exercise 2.4

<hr/>
## 3. Forecasts

We forecast both signals with the protocol of Lab 036. At every test origin $k=2999,\dots,3949$ a model reads at most the window of the last $L=64$ values and returns the leads $\tau=1,\dots,50$ at once, one output per lead. Four models are static, they map a fixed window of past values to the future. Persistence repeats the last value, AR and NVAR are ridge regressions on the delay vector and on the delay vector with its quadratic monomials, and the MA model forecasts from the past one-step prediction errors (Lecture 7, Lab 034). The RNN and the LSTM network carry a hidden state through the window and learn their recurrence by gradient descent (Lecture 8, Lab 036). For the MSO signal, persistence and the direct AR model were computed in Exercise 1.5.

### Exercise 3.1. Persistence, AR and NVAR

Compute persistence, the direct AR model and the direct NVAR model for Rössler $x$ with `ridge_model` of Lab 036 (order and ridge strength chosen by the validation $C_{\rm tot}$), and the direct NVAR model for the MSO signal. Rebuild the selected Rössler NVAR model with ReservoirPy, `NVAR(delay=p, order=2, strides=1) >> Ridge(ridge=beta)`, fitted on the same origins with `warmup=63` as in Lab 034, and check that its test predictions agree with the JAX model.

In [ ]:
# your code: persistence, direct AR and NVAR with validation selection, ReservoirPy NVAR check

### Exercise 3.2. Moving-average model

The MA($q$) model writes the signal as its mean plus a weighted sum of the present and the last $q$ innovations, $s_k=\mu+\varepsilon_k+\sum_{j=1}^{q}\theta_j\varepsilon_{k-j}$. Fit MA(8) to the first 3000 samples of each signal by maximum likelihood with `statsmodels`, `ARIMA(order=(0, 0, 8), trend="c")`, as in Lab 036. Compute the one-step prediction errors $e_k=s_k-\hat s_k$ of the fitted model over the whole record with `apply`, and forecast with
$$
\hat s_{k+\tau}=\mu+\sum_{j=\tau}^{q}\theta_j\,e_{k+\tau-j},
$$
which uses only errors up to the origin $k$. Compare the forecast from the first test origin with the `forecast` method of `statsmodels`. Report $\mu$, the coefficients $\theta_j$, the smallest modulus of the roots of the MA polynomial $1+\sum_j\theta_jz^j$, $C(1)$, $C(8)$ and $C_{\rm tot}$, and explain the value of $C(\tau)$ for $\tau>8$.

In [ ]:
# your code: ARIMA(0, 0, 8) fit on the training block, one-step errors over the record, MA forecasts at the test origins,
# check against statsmodels forecast, scores for both signals

### Exercise 3.3. Recurrent networks

Train an RNN and an LSTM network in `flax.nnx` as in Lab 036. The RNN updates its hidden state $\mathbf h_k\in\mathbb R^{n_h}$ by $\mathbf h_k=\tanh(W_h\mathbf h_{k-1}+W_xs_k+\mathbf b)$. The LSTM adds a cell state $\mathbf c_k$ controlled by three gates, with the sigmoid $\sigma(a)=1/(1+e^{-a})$, the elementwise product $\odot$ and the concatenation $[\mathbf h_{k-1},s_k]$,
$$
\begin{aligned}
\mathbf f_k&=\sigma\big(W_f[\mathbf h_{k-1},s_k]+\mathbf b_f\big), &
\mathbf i_k&=\sigma\big(W_i[\mathbf h_{k-1},s_k]+\mathbf b_i\big),\\
\mathbf g_k&=\tanh\big(W_g[\mathbf h_{k-1},s_k]+\mathbf b_g\big), &
\mathbf o_k&=\sigma\big(W_o[\mathbf h_{k-1},s_k]+\mathbf b_o\big),\\
\mathbf c_k&=\mathbf f_k\odot\mathbf c_{k-1}+\mathbf i_k\odot\mathbf g_k, &
\mathbf h_k&=\mathbf o_k\odot\tanh(\mathbf c_k),
\end{aligned}
$$
the forget gate $\mathbf f_k$, the input gate $\mathbf i_k$, the candidate $\mathbf g_k$ and the output gate $\mathbf o_k$, with the initial forget bias $\mathbf b_f=1$. Both networks have $n_h=32$ hidden units, run over the window of $L=64$ values with `jax.lax.scan` from $\mathbf h=0$, and read the 50 leads from the last hidden state with a linear layer. Train them on both signals with the loop `fit_network` of Lab 036 (minibatches of 128 windows, Adam with learning rate $5\times10^{-3}$, gradient clipping at norm 1, early stopping with patience 20, at most 250 epochs, seed 0). Check the number of parameters against the formula, and report the number of epochs, the training time, the best validation loss and the test $C_{\rm tot}$. Plot the training and validation losses.

In [ ]:
# your code: RNNCell, LSTMCell and Recurrent modules in flax.nnx, training loop of Lab 036, both signals, loss curves

### Exercise 3.4. One-step forecasts

Plot for every model the forecast one step ahead, $\tau=1$, in the style of Labs 034 and 036, with one column per signal. The upper panels show the target $s_{k+1}$ and the forecast $\hat s_{k+1}$ on the first 100 test targets, and the lower panels show $\hat s_{k+1}$ against $s_{k+1}$ on all test targets, with $C(1)=r(1)^2$ and the one-step NMSE. Describe what each figure shows about the model.

In [ ]:
# your code: one figure per model (persistence, AR, MA, NVAR, RNN, LSTM), columns MSO and Rossler,
# upper row target and forecast on the first 100 test steps, lower row forecast against target with C and NMSE

### Exercise 3.5. Capacity curves and interpretation

Plot the capacity curves $C(\tau)=r(\tau)^2$, $\tau=1,\dots,50$, of the six models for both signals, and tabulate $C_{\rm tot}=\sum_\tau C(\tau)$ for both signals, together with $\tau_{1/2}$ and the NMSE at $\tau=1$ and $\tau=50$ for Rössler $x$. Plot also the one-step return map of the maxima, $x_{\max,j+1}$ against $x_{\max,j}$, of successive local maxima of the rescaled Rössler signal. Read the results against the characterisation. Express the longest lead in Lyapunov times and state what this implies for the attainable skill. Compare the static models with the recurrent networks, explain the gap between the linear and the nonlinear models with the property of the attractor visible in $D_{\rm KY}$, in the spectrum and in the return map, and compare with the MSO signal and with the chaotic signals of Labs 034 and 036.

In [ ]:
# your code: C(tau) of all models for both signals, C_tot table, return map of successive maxima, interpretation

### Optional exercise 3.6. Echo state network

Forecast Rössler $x$ with the ESN of Lab 036, a fixed random recurrent network whose state is read by a ridge readout. Use one fixed random reservoir of $N=300$ units with 10% nonzero couplings (seed 0), driven by the rescaled signal from the first sample, and choose the spectral radius $\rho\in\{0.5,0.8,0.95,1.1\}$, the input scaling $\sigma_{\rm in}\in\{0.1,0.5,1\}$, the leak rate $\alpha\in\{0.3,1\}$ and the ridge strength $\beta\in\{10^{-8},10^{-6},10^{-4},10^{-2}\}$ by the validation $C_{\rm tot}$. Rebuild the selected model with ReservoirPy, `Reservoir(W=..., Win=..., bias=0, lr=alpha) >> Ridge(ridge=beta)`, compare its readout predictions with the JAX version, and place the test $C_{\rm tot}$ of the ESN in the table of Exercise 3.5.

In [ ]:
# your code: ESN in JAX with the grid search of Lab 036, ReservoirPy check